# GPT-generated summaries of category contents

In [1]:
import pandas as pd
from discovery_child_development import PROJECT_DIR, S3_BUCKET
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [2]:
OUTPUTS_DIR = ENRICHED_DATA_DIR / 'themes'
OUTPUTS_DIR.mkdir(exist_ok=True, parents=True)

In [3]:
from discovery_child_development.utils.openai_utils import client

In [13]:
import utils
topics_df = utils.load_topic_data()

# OpenAlex themes

In [129]:
openalex_df = pd.read_csv(PROJECT_DIR / "outputs/data/tables/openalex_final.csv")

In [130]:
from discovery_child_development.getters.openalex import get_sentence_embeddings

# Path to sentence embeddings
VECTORS_PATH = "data/outputs/vectors/"
VECTORS_FILE_1 = "sentence_vectors_openalex_384_labelled.parquet"
VECTORS_FILE_2 = "sentence_vectors_384_labelled.parquet"

In [131]:
# Load dataset sentence embeddings (all-MiniLM-L6-v2)
embeddings_1 = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET,
        filepath=VECTORS_PATH,
        filename=VECTORS_FILE_1,
        id="id",
    )
    .reset_index()
    # Simplify the id by removing https
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    # .set_index("id")
)

In [132]:
# Load dataset sentence embeddings (all-MiniLM-L6-v2)
embeddings_2 = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET,
        filepath=VECTORS_PATH,
        filename=VECTORS_FILE_2,
        id="id",
    )
    .reset_index()
    # Simplify the id by removing https
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    # .set_index("id")
)

In [133]:
embeddings_all = pd.concat([embeddings_1, embeddings_2], ignore_index=True).drop_duplicates('id').set_index('id')

In [134]:
len(embeddings_all)

83409

In [135]:
len(openalex_df)

69748

## Pre-process data

In [136]:
def get_overlaps(df: pd.DataFrame, categories: list, category_column: str) -> pd.DataFrame:
    """Fetch the datapoints that have all the categories in the list

    Args:
        df (pd.DataFrame): DataFrame containing the datapoints
        categories (list): List of categories
        category_column (str): Column name that contains the categories

    Returns:
        pd.DataFrame: DataFrame containing the datapoints that have all the categories in the list
    """
    return (
        df
        .copy()
        # transform comma separated string to list, account for nulls
        .assign(**{category_column: lambda x: x[category_column].fillna('').str.split(', ')})
        # filter the rows that have all the categories in the list
        .loc[lambda x: x[category_column].apply(lambda y: set(categories).issubset(y))]
    )

In [137]:
for topic1 in ['Mobile', 'Internet', 'Immersive tech']:
    themes = []
    topics2 = list(topics_df.subtype.unique())

    # for topic2 in topics2[0:1]: 
    for topic2 in topics2: 

        # for dataset in ["Publications", "Patents"]:

        # Get the documents that have both topics
        _df = (
            get_overlaps(openalex_df, [topic1, topic2], 'minor_category')[['id', 'text', 'minor_category']]
            # .query("Dataset == @dataset")
            .assign(text_ = lambda x: 'ID: ' + x['id'] + ' | TEXT: ' + x['text'])
        )
        # Take 50 documents which might be most focussed on the two topics
        df = (
            _df
            # shuffle
            .sample(len(_df))
            # sort by number of topics (to get the more specific examples first)
            .assign(n_topics = lambda df: df['minor_category'].apply(lambda x: len(x)))
            .sort_values('n_topics')
            # get the first 50
            .head(50)
        )

        topic1_name = topics_df.query("subtype == @topic1").iloc[0]["subtype"]
        topic2_name = topics_df.query("subtype == @topic2").iloc[0]["subtype"]

        topic1_type = topics_df.query("subtype == @topic1").iloc[0]["type"]
        topic2_type = topics_df.query("subtype == @topic2").iloc[0]["type"]

        abstract_texts = "\n\n".join(df.text_.to_list())

        gpt_message = f"You are an expert researcher on early childhood development. \
            Here are documents related to topics: Topic 1: {topic1_name} ({topic1_type}) and Topic 2: {topic2_name} ({topic2_type}). \
            Detect one to three distinct themes across these documents, that are closely related to the interaction between these two topics, and summarise these themes. \
            For each theme, write up to two sentences of summary, highlighting how the two topics overlap \
            and also three or four most interesting examples of innovations, technologies or interventions, referencing the alphanumeric ID of the document describing the example. \
            Focus on themes and examples that consider both topics together. \
            Here's an example: \n\n##Example\n\nTheme: [Theme name]\nSummary:[Summary of the theme]\nExample 1: \
            [Example of innovation, technology or intervention] (ID: [ID of the document] \nExample 2: [Example of innovation, technology or intervention] \
            (ID: [ID of the document])\n\n##Document texts\n\n {abstract_texts} \n\n##Themes\n\n"
        
        # Generate cluster descriptions
        print(f"Generating cluster themes for {topic1_name} and {topic2_name}")
        print(f"Number of total documents: {len(_df)}")
        print(f"Number of documents used: {len(df)}")
        messages = [
            {
                "role": "user",
                "content": gpt_message,
            }
        ]
        if len(df) > 0:
            chatgpt_output = client.chat.completions.create(
                # model="gpt-4-turbo-2024-04-09",
                model = "gpt-4o-2024-05-13",
                messages=messages,
                temperature=0.6,
                max_tokens=2000,
            )
            cluster_themes = chatgpt_output.choices[0].message.content
        else:
            print("No documents found for the given topics") 

        themes.append(
            {
                "topic1": topic1,
                "topic2": topic2,
                "dataset": 'openalex',
                "numb_docs": len(df),
                "total_number": len(_df),
                "cluster_themes": cluster_themes,
            }
        )

    themes_df = pd.DataFrame(themes).rename(columns={'numb_docs': 'n_docs'})
    # if numb_docs = 0, then make cluster_themes empty
    themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

    # Save json and save csv
    themes_df.to_csv(OUTPUTS_DIR / f'openalex_topic_themes_{topic1}.csv', index=False)
    themes_df.to_json(OUTPUTS_DIR / f'openalex_topic_themes_{topic1}.json', orient='records')
        

Generating cluster themes for Mobile and Genetics
Number of total documents: 8
Number of documents used: 8
2024-06-17 22:48:51,910 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Neuroscience
Number of total documents: 47
Number of documents used: 47
2024-06-17 22:49:02,770 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Operations
Number of total documents: 22
Number of documents used: 22
2024-06-17 22:49:18,612 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Preschool
Number of total documents: 190
Number of documents used: 50
2024-06-17 22:49:27,835 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Cognitive development
Number of t

In [116]:
themes_df = pd.DataFrame(themes).rename(columns={'numb_docs': 'n_docs'})
# if numb_docs = 0, then make cluster_themes empty
themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

# Save json and save csv
themes_df.to_csv(OUTPUTS_DIR / f'openalex_topic_themes_{topic1}.csv', index=False)
themes_df.to_json(OUTPUTS_DIR / f'openalex_topic_themes_{topic1}.json', orient='records')

# UKRI

In [138]:
gtr_df = pd.read_csv(PROJECT_DIR / "outputs/data/tables/gtr_final.csv")
gtr_df = gtr_df.assign(identifier = lambda df: df['url'].apply(lambda x: x.replace("https://gtr.ukri.org/projects?ref=","")))

In [139]:
from discovery_child_development.getters.openalex import get_sentence_embeddings

# Path to sentence embeddings
VECTORS_PATH = "data/outputs/vectors/"
VECTORS_FILE = "sentence_vectors_gtr_384_labelled.parquet"

# Load dataset sentence embeddings (all-MiniLM-L6-v2)
embeddings_all = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET,
        filepath=VECTORS_PATH,
        filename=VECTORS_FILE,
        id="id",
    )
    .reset_index()
    # Simplify the id by removing https
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    .set_index("id")
)

In [140]:
len(embeddings_all)

1093

In [141]:
themes_gtr = []

In [142]:
for topic1 in ['Mobile', 'Internet', 'Immersive tech']:
    themes_gtr = []
    topics2 = list(topics_df.subtype.unique())

    # for topic2 in topics2[0:1]: 
    for topic2 in topics2: 

        # for dataset in ["Publications", "Patents"]:

        # Get the documents that have both topics
        _df = (
            get_overlaps(gtr_df, [topic1, topic2], 'minor_category')[['id', 'text', 'minor_category', 'identifier']]
            # .query("Dataset == @dataset")
            .assign(text_ = lambda x: 'ID: ' + x['identifier'] + ' | TEXT: ' + x['text'])
        )
        # Take 50 documents which might be most focussed on the two topics
        df = (
            _df
            # shuffle
            .sample(len(_df))
            # sort by number of topics (to get the more specific examples first)
            .assign(n_topics = lambda df: df['minor_category'].apply(lambda x: len(x)))
            .sort_values('n_topics')
            # get the first 50
            .head(50)
        )

        topic1_name = topics_df.query("subtype == @topic1").iloc[0]["subtype"]
        topic2_name = topics_df.query("subtype == @topic2").iloc[0]["subtype"]

        topic1_type = topics_df.query("subtype == @topic1").iloc[0]["type"]
        topic2_type = topics_df.query("subtype == @topic2").iloc[0]["type"]

        abstract_texts = "\n\n".join(df.text_.to_list())

        gpt_message = f"You are an expert researcher on early childhood development. \
            Here are documents related to topics: Topic 1: {topic1_name} ({topic1_type}) and Topic 2: {topic2_name} ({topic2_type}). \
            Detect one to three distinct themes across these documents, that are closely related to the interaction between these two topics, and summarise these themes. \
            For each theme, write up to two sentences of summary, highlighting how the two topics overlap \
            and also three or four most interesting examples of innovations, technologies or interventions, referencing the alphanumeric ID of the document describing the example. \
            Focus on themes and examples that consider both topics together. \
            Here's an example: \n\n##Example\n\nTheme: [Theme name]\nSummary:[Summary of the theme]\nExample 1: \
            [Example of innovation, technology or intervention] (ID: [ID of the document] \nExample 2: [Example of innovation, technology or intervention] \
            (ID: [ID of the document])\n\n##Document texts\n\n {abstract_texts} \n\n##Themes\n\n"
        
        # Generate cluster descriptions
        print(f"Generating cluster themes for {topic1_name} and {topic2_name}")
        print(f"Number of total documents: {len(_df)}")
        print(f"Number of documents used: {len(df)}")
        messages = [
            {
                "role": "user",
                "content": gpt_message,
            }
        ]
        if len(df) > 0:
            chatgpt_output = client.chat.completions.create(
                # model="gpt-4-turbo-2024-04-09",
                model = "gpt-4o-2024-05-13",
                messages=messages,
                temperature=0.6,
                max_tokens=2000,
            )
            cluster_themes = chatgpt_output.choices[0].message.content
        else:
            print("No documents found for the given topics") 

        themes_gtr.append(
            {
                "topic1": topic1,
                "topic2": topic2,
                "dataset": 'gtr',
                "numb_docs": len(df),
                "total_number": len(_df),
                "cluster_themes": cluster_themes,
            }
        )
        
    themes_df = pd.DataFrame(themes_gtr).rename(columns={'numb_docs': 'n_docs'})
    # if numb_docs = 0, then make cluster_themes empty
    themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

    # Save json and save csv
    themes_df.to_csv(OUTPUTS_DIR / f'gtr_topic_themes_{topic1}.csv', index=False)
    themes_df.to_json(OUTPUTS_DIR / f'gtr_topic_themes_{topic1}.json', orient='records')        

Generating cluster themes for Mobile and Genetics
Number of total documents: 0
Number of documents used: 0
No documents found for the given topics
Generating cluster themes for Mobile and Neuroscience
Number of total documents: 4
Number of documents used: 4
2024-06-17 23:12:16,741 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Operations
Number of total documents: 3
Number of documents used: 3
2024-06-17 23:12:28,925 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Preschool
Number of total documents: 3
Number of documents used: 3
2024-06-17 23:12:35,585 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Cognitive development
Number of total documents: 6
Number of documents used: 6
2024-06-17 23:12:43,980 - httpx - INFO - H

In [122]:
themes_df = pd.DataFrame(themes_gtr).rename(columns={'numb_docs': 'n_docs'})
# if numb_docs = 0, then make cluster_themes empty
themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

# Save json and save csv
themes_df.to_csv(OUTPUTS_DIR / f'gtr_topic_themes_{topic1}.csv', index=False)
themes_df.to_json(OUTPUTS_DIR / f'gtr_topic_themes_{topic1}.json', orient='records')

# Crunchbase

In [158]:
crunchbase_df = pd.read_csv(PROJECT_DIR / "outputs/data/tables/crunchbase_final.csv")

In [159]:
from discovery_child_development.getters.openalex import get_sentence_embeddings

# Path to sentence embeddings
VECTORS_PATH = "data/outputs/vectors/"
VECTORS_FILE = "sentence_vectors_crunchbase_384_labelled.parquet"

# Load dataset sentence embeddings (all-MiniLM-L6-v2)
embeddings_all = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET,
        filepath=VECTORS_PATH,
        filename=VECTORS_FILE,
        id="id",
    )
    .reset_index()
    # Simplify the id by removing https
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    .set_index("id")
)

In [145]:
len(embeddings_all)

8696

In [160]:
themes_cb = []

In [161]:
for topic1 in ['AI', 'Mobile', 'Internet', 'Immersive tech']:
    themes_cb = []
    topics2 = list(topics_df.subtype.unique())

    # for topic2 in topics2[0:1]: 
    for topic2 in topics2: 

        # for dataset in ["Publications", "Patents"]:

        # Get the documents that have both topics
        _df = (
            get_overlaps(crunchbase_df, [topic1, topic2], 'minor_category')[['id', 'text', 'minor_category', 'url']]
            # .query("Dataset == @dataset")
            .assign(text_ = lambda x: 'ID: ' + x['url'] + ' | TEXT: ' + x['text'])
        )
        # Take 50 documents which might be most focussed on the two topics
        df = (
            _df
            # shuffle
            .sample(len(_df))
            # sort by number of topics (to get the more specific examples first)
            .assign(n_topics = lambda df: df['minor_category'].apply(lambda x: len(x)))
            .sort_values('n_topics')
            # get the first 50
            .head(50)
        )

        topic1_name = topics_df.query("subtype == @topic1").iloc[0]["subtype"]
        topic2_name = topics_df.query("subtype == @topic2").iloc[0]["subtype"]

        topic1_type = topics_df.query("subtype == @topic1").iloc[0]["type"]
        topic2_type = topics_df.query("subtype == @topic2").iloc[0]["type"]

        abstract_texts = "\n\n".join([t for t in df.text_.to_list() if isinstance(t, str)])

        gpt_message = f"You are an expert researcher on early childhood development. \
            Here are documents related to topics: Topic 1: {topic1_name} ({topic1_type}) and Topic 2: {topic2_name} ({topic2_type}). \
            Detect one to three distinct themes across these documents, that are closely related to the interaction between these two topics, and summarise these themes. \
            For each theme, write up to two sentences of summary, highlighting how the two topics overlap \
            and also three or four most interesting examples of innovations, technologies or interventions, referencing the alphanumeric ID of the document describing the example. \
            Focus on themes and examples that consider both topics together. \
            Here's an example: \n\n##Example\n\nTheme: [Theme name]\nSummary:[Summary of the theme]\nExample 1: \
            [Example of innovation, technology or intervention] (ID: [ID of the document] \nExample 2: [Example of innovation, technology or intervention] \
            (ID: [ID of the document])\n\n##Document texts\n\n {abstract_texts} \n\n##Themes\n\n"
        
        # Generate cluster descriptions
        print(f"Generating cluster themes for {topic1_name} and {topic2_name}")
        print(f"Number of total documents: {len(_df)}")
        print(f"Number of documents used: {len(df)}")
        messages = [
            {
                "role": "user",
                "content": gpt_message,
            }
        ]
        if len(df) > 0:
            chatgpt_output = client.chat.completions.create(
                # model="gpt-4-turbo-2024-04-09",
                model = "gpt-4o-2024-05-13",
                messages=messages,
                temperature=0.6,
                max_tokens=2000,
            )
            cluster_themes = chatgpt_output.choices[0].message.content
        else:
            print("No documents found for the given topics") 

        themes_cb.append(
            {
                "topic1": topic1,
                "topic2": topic2,
                "dataset": 'crunchbase',
                "numb_docs": len(df),
                "total_number": len(_df),
                "cluster_themes": cluster_themes,
            }
        )

    themes_df = pd.DataFrame(themes_cb).rename(columns={'numb_docs': 'n_docs'})
    # if numb_docs = 0, then make cluster_themes empty
    themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

    # Save json and save csv
    themes_df.to_csv(OUTPUTS_DIR / f'crunchbase_topic_themes_{topic1}.csv', index=False)
    themes_df.to_json(OUTPUTS_DIR / f'crunchbase_topic_themes_{topic1}.json', orient='records')        
        

Generating cluster themes for AI and Genetics
Number of total documents: 1
Number of documents used: 1
2024-06-18 07:50:01,476 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Neuroscience
Number of total documents: 11
Number of documents used: 11
2024-06-18 07:50:11,879 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Operations
Number of total documents: 16
Number of documents used: 16
2024-06-18 07:50:23,796 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Preschool
Number of total documents: 8
Number of documents used: 8
2024-06-18 07:50:32,040 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Cognitive development
Number of total documents: 31
Numb

In [ ]:
themes_df = pd.DataFrame(themes_cb).rename(columns={'numb_docs': 'n_docs'})
# if numb_docs = 0, then make cluster_themes empty
themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

# Save json and save csv
themes_df.to_csv(OUTPUTS_DIR / f'crunchbase_topic_themes_{topic1}.csv', index=False)
themes_df.to_json(OUTPUTS_DIR / f'crunchbase_topic_themes_{topic1}.json', orient='records')

## Patents

In [156]:
patents_df = pd.read_csv(PROJECT_DIR / "outputs/data/tables/patents_final.csv").dropna(subset=['minor_category'])

In [157]:
for topic1 in ['AI', 'Mobile', 'Internet', 'Immersive tech']:
    themes_patents = []
    topics2 = list(topics_df.subtype.unique())

    # for topic2 in topics2[0:1]: 
    for topic2 in topics2: 

        # for dataset in ["Publications", "Patents"]:

        # Get the documents that have both topics
        _df = (
            get_overlaps(patents_df, [topic1, topic2], 'minor_category')[['id', 'text', 'minor_category']]
            # .query("Dataset == @dataset")
            .assign(text_ = lambda x: 'ID: ' + x['id'] + ' | TEXT: ' + x['text'])
        )
        # Take 50 documents which might be most focussed on the two topics
        df = (
            _df
            # shuffle
            .sample(len(_df))
            # sort by number of topics (to get the more specific examples first)
            .assign(n_topics = lambda df: df['minor_category'].apply(lambda x: len(x)))
            .sort_values('n_topics')
            # get the first 50
            .head(50)
        )

        topic1_name = topics_df.query("subtype == @topic1").iloc[0]["subtype"]
        topic2_name = topics_df.query("subtype == @topic2").iloc[0]["subtype"]

        topic1_type = topics_df.query("subtype == @topic1").iloc[0]["type"]
        topic2_type = topics_df.query("subtype == @topic2").iloc[0]["type"]

        abstract_texts = "\n\n".join([t for t in df.text_.to_list() if isinstance(t, str)])

        gpt_message = f"You are an expert researcher on early childhood development. \
            Here are documents related to topics: Topic 1: {topic1_name} ({topic1_type}) and Topic 2: {topic2_name} ({topic2_type}). \
            Detect one to three distinct themes across these documents, that are closely related to the interaction between these two topics, and summarise these themes. \
            For each theme, write up to two sentences of summary, highlighting how the two topics overlap \
            and also three or four most interesting examples of innovations, technologies or interventions, referencing the alphanumeric ID of the document describing the example. \
            Focus on themes and examples that consider both topics together. \
            Here's an example: \n\n##Example\n\nTheme: [Theme name]\nSummary:[Summary of the theme]\nExample 1: \
            [Example of innovation, technology or intervention] (ID: [ID of the document] \nExample 2: [Example of innovation, technology or intervention] \
            (ID: [ID of the document])\n\n##Document texts\n\n {abstract_texts} \n\n##Themes\n\n"
        
        # Generate cluster descriptions
        print(f"Generating cluster themes for {topic1_name} and {topic2_name}")
        print(f"Number of total documents: {len(_df)}")
        print(f"Number of documents used: {len(df)}")
        messages = [
            {
                "role": "user",
                "content": gpt_message,
            }
        ]
        if len(df) > 0:
            chatgpt_output = client.chat.completions.create(
                # model="gpt-4-turbo-2024-04-09",
                model = "gpt-4o-2024-05-13",
                messages=messages,
                temperature=0.6,
                max_tokens=2000,
            )
            cluster_themes = chatgpt_output.choices[0].message.content
        else:
            print("No documents found for the given topics") 

        themes_patents.append(
            {
                "topic1": topic1,
                "topic2": topic2,
                "dataset": 'crunchbase',
                "numb_docs": len(df),
                "total_number": len(_df),
                "cluster_themes": cluster_themes,
            }
        )

    themes_df = pd.DataFrame(themes_patents).rename(columns={'numb_docs': 'n_docs'})
    # if numb_docs = 0, then make cluster_themes empty
    themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

    # Save json and save csv
    themes_df.to_csv(OUTPUTS_DIR / f'patents_topic_themes_{topic1}.csv', index=False)
    themes_df.to_json(OUTPUTS_DIR / f'patents_topic_themes_{topic1}.json', orient='records')        
        

Generating cluster themes for AI and Genetics
Number of total documents: 8
Number of documents used: 8
2024-06-18 07:23:32,252 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Neuroscience
Number of total documents: 92
Number of documents used: 50
2024-06-18 07:23:49,555 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Operations
Number of total documents: 12
Number of documents used: 12
2024-06-18 07:23:57,852 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Preschool
Number of total documents: 42
Number of documents used: 42
2024-06-18 07:24:06,160 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for AI and Cognitive development
Number of total documents: 7
Num

### Process the tables

In [173]:
for dataset in ['openalex', 'patents', 'gtr', 'crunchbase']:
    for topic in ['AI', 'Mobile', 'Internet', 'Immersive tech']:
        print(f"Loading {dataset} {topic} themes")
        df = (
            pd.read_csv(PROJECT_DIR / f"outputs/enrichments/themes/{dataset}_topic_themes_{topic}.csv")
            .merge(topics_df[['type', 'subtype']], left_on='topic2', right_on='subtype')
            .rename(columns={'topic1': 'minor_category_1', 'topic2': 'minor_category_2', 'type': 'major_category_2', 'cluster_themes': 'themes'})
            .drop('subtype', axis=1)
        )[['dataset', 'minor_category_1', 'minor_category_2', 'major_category_2', 'n_docs', 'total_number', 'themes']]
        df.to_csv(PROJECT_DIR / f"outputs/enrichments/themes/_{dataset}_topic_themes_{topic}.csv", index=False)

Loading openalex AI themes
Loading openalex Mobile themes
Loading openalex Internet themes
Loading openalex Immersive tech themes
Loading patents AI themes
Loading patents Mobile themes
Loading patents Internet themes
Loading patents Immersive tech themes
Loading gtr AI themes
Loading gtr Mobile themes
Loading gtr Internet themes
Loading gtr Immersive tech themes
Loading crunchbase AI themes
Loading crunchbase Mobile themes
Loading crunchbase Internet themes
Loading crunchbase Immersive tech themes


In [164]:
df

,topic1,topic2,dataset,n_docs,total_number,cluster_themes,type,subtype
0,AI,Genetics,openalex,50,55,## Theme: Integration of AI and Genetic Analys...,Biosciences,Genetics
1,AI,Neuroscience,openalex,50,94,## Theme: Integration of AI and Neuroscience i...,Biosciences,Neuroscience
2,AI,Operations,openalex,4,4,## Theme: Enhancing Childcare through AI-drive...,Child care & preschool,Operations
3,AI,Preschool,openalex,15,15,##Theme: Development of Cognitive and Algorith...,Child care & preschool,Preschool
4,AI,Cognitive development,openalex,50,71,## Theme: AI-Enhanced Early Childhood Educatio...,Development & learning,Cognitive development
5,AI,Communication and language,openalex,45,45,##Theme: Early Detection and Diagnosis of Lang...,Development & learning,Communication and language
6,AI,Expressive arts and design,openalex,37,37,##Theme: Integration of AI in Expressive Arts ...,Development & learning,Expressive arts and design
7,AI,Literacy,openalex,11,11,### Theme: Early AI Literacy and Cognitive Dev...,Development & learning,Literacy
8,AI,Mathematics,openalex,28,28,## Theme: AI-Enhanced Personalized Learning in...,Development & learning,Mathematics
9,AI,Personal social emotional,openalex,8,8,##Theme: AI-Enhanced Socio-Emotional Learning\...,Development & learning,Personal social emotional
